In [ ]:
import pandas as pd
import numpy as np
import joblib
import time
import json
import os

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.model_selection import cross_val_score, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")

###  load  the  preprocess data sets    

In [ ]:
X_train = pd.read_csv("../data/X_train.csv")
X_test = pd.read_csv("../data/X_test.csv")
y_train = pd.read_csv("../data/y_train.csv").squeeze()
y_test = pd.read_csv("../data/y_test.csv").squeeze()

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (799815, 27)
X_test: (199954, 27)


###  hyperparameter  tunning    

In [ ]:
# Tuning on the full training set would be slow; we tune on a representative
# 50,000-row sample, then fit the best configuration on the full training data.
tune_sample = X_train.sample(50000, random_state=42)
y_tune_sample = y_train.loc[tune_sample.index]

print("Tuning sample shape:", tune_sample.shape)

Tuning sample shape: (50000, 27)


### Define a results tracker   

In [ ]:
results = {}
trained_models = {}

def evaluate_model(name, model, X_tr, y_tr, X_te, y_te, train_time):
    start = time.time()
    y_pred = model.predict(X_te)
    predict_time = time.time() - start

    mae = mean_absolute_error(y_te, y_pred)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))
    r2 = r2_score(y_te, y_pred)

    cv_sample = X_tr.sample(20000, random_state=42)
    cv_scores = cross_val_score(model, cv_sample, y_tr.loc[cv_sample.index],
                                 cv=5, scoring="r2", n_jobs=-1)

    results[name] = {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "CV_R2_Mean": cv_scores.mean(),
        "CV_R2_Std": cv_scores.std(),
        "Train_Time_Sec": train_time,
        "Predict_Time_Sec": predict_time
    }
    trained_models[name] = model

    print(f"{name} — MAE: {mae:.4f} | RMSE: {rmse:.4f} | R²: {r2:.4f} | CV R²: {cv_scores.mean():.4f}")
    return model

### Model 1: Linear Regression   

In [ ]:
start = time.time()
lr = LinearRegression()
lr.fit(X_train, y_train)
train_time = time.time() - start

lr = evaluate_model("Linear Regression", lr, X_train, y_train, X_test, y_test, train_time)

Linear Regression — MAE: 0.3983 | RMSE: 0.4993 | R²: 0.9132 | CV R²: 0.9121


### Model 2: Decision Tree

In [ ]:
dt_params = {
    "max_depth": [5, 8, 10, 15, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

dt_search = RandomizedSearchCV(DecisionTreeRegressor(random_state=42), dt_params,
                                n_iter=10, cv=3, scoring="r2", random_state=42, n_jobs=-1)
dt_search.fit(tune_sample, y_tune_sample)
print("Best Decision Tree params:", dt_search.best_params_)

start = time.time()
dt = DecisionTreeRegressor(**dt_search.best_params_, random_state=42)
dt.fit(X_train, y_train)
train_time = time.time() - start

dt = evaluate_model("Decision Tree", dt, X_train, y_train, X_test, y_test, train_time)

Best Decision Tree params: {'min_samples_split': 10, 'min_samples_leaf': 4, 'max_depth': 8}
Decision Tree — MAE: 0.4057 | RMSE: 0.5083 | R²: 0.9101 | CV R²: 0.9022


### Model 3: Random Forest

In [14]:
rf_params = {
    "n_estimators": [50, 100, 150],       # reduced from [100, 200, 300]
    "max_depth": [10, 15, 20],             # removed None (unlimited depth = slowest)
    "min_samples_split": [5, 10],
    "min_samples_leaf": [2, 4]
}

rf_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=4),  # capped at 4 cores instead of -1 (all cores)
    rf_params, n_iter=6, cv=3, scoring="r2", random_state=42, n_jobs=1  # search itself not parallel
)
rf_search.fit(tune_sample, y_tune_sample)
print("Best Random Forest params:", rf_search.best_params_)

start = time.time()
rf = RandomForestRegressor(**rf_search.best_params_, random_state=42, n_jobs=4)
rf.fit(X_train.sample(300000, random_state=42), y_train.loc[X_train.sample(300000, random_state=42).index])  # subsample instead of full 800K
train_time = time.time() - start

rf = evaluate_model("Random Forest", rf, X_train, y_train, X_test, y_test, train_time)

KeyboardInterrupt: 

#### Model 4: Gradient Boosting   

In [ ]:
gb_params = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 5, 7],
    "subsample": [0.8, 1.0]
}

gb_search = RandomizedSearchCV(GradientBoostingRegressor(random_state=42), gb_params,
                                n_iter=10, cv=3, scoring="r2", random_state=42, n_jobs=-1)
gb_search.fit(tune_sample, y_tune_sample)
print("Best Gradient Boosting params:", gb_search.best_params_)

start = time.time()
gb = GradientBoostingRegressor(**gb_search.best_params_, random_state=42)
gb.fit(X_train, y_train)
train_time = time.time() - start

gb = evaluate_model("Gradient Boosting", gb, X_train, y_train, X_test, y_test, train_time)

### Model 5: XGBoost

In [ ]:
xgb_params = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 5, 7, 9],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

xgb_search = RandomizedSearchCV(
    XGBRegressor(random_state=42, tree_method="hist", device="cuda"),
    xgb_params, n_iter=10, cv=3, scoring="r2", random_state=42, n_jobs=1
)
xgb_search.fit(tune_sample, y_tune_sample)
print("Best XGBoost params:", xgb_search.best_params_)

start = time.time()
xgb_model = XGBRegressor(**xgb_search.best_params_, random_state=42, tree_method="hist", device="cuda")
xgb_model.fit(X_train, y_train)
train_time = time.time() - start

xgb_model = evaluate_model("XGBoost", xgb_model, X_train, y_train, X_test, y_test, train_time)

Best XGBoost params: {'subsample': 0.8, 'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.1, 'colsample_bytree': 0.8}
XGBoost — MAE: 0.3993 | RMSE: 0.5005 | R²: 0.9128 | CV R²: 0.9084


### Model 6: LightGBM    

In [ ]:
lgbm_params = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 5, 7, -1],
    "num_leaves": [20, 31, 50],
    "subsample": [0.8, 1.0]
}

lgbm_search = RandomizedSearchCV(
    LGBMRegressor(random_state=42, verbose=-1, n_jobs=4),
    lgbm_params, n_iter=10, cv=3, scoring="r2", random_state=42, n_jobs=1
)
lgbm_search.fit(tune_sample, y_tune_sample)
print("Best LightGBM params:", lgbm_search.best_params_)

start = time.time()
lgbm_model = LGBMRegressor(**lgbm_search.best_params_, random_state=42, verbose=-1, n_jobs=4)
lgbm_model.fit(X_train, y_train)
train_time = time.time() - start

lgbm_model = evaluate_model("LightGBM", lgbm_model, X_train, y_train, X_test, y_test, train_time)

LightGBM GPU not available, falling back to CPU: GPU Tree Learner was not enabled in this build.
Please recompile with CMake option -DUSE_GPU=1
Best LightGBM params: {'subsample': 1.0, 'num_leaves': 31, 'n_estimators': 100, 'max_depth': 7, 'learning_rate': 0.05}
LightGBM — MAE: 0.3988 | RMSE: 0.4999 | R²: 0.9130 | CV R²: 0.9103


### Model 7: CatBoost    

In [ ]:
cat_params = {
    "iterations": [200, 300, 500],
    "learning_rate": [0.01, 0.05, 0.1],
    "depth": [4, 6, 8],
    "l2_leaf_reg": [1, 3, 5]
}

cat_search = RandomizedSearchCV(
    CatBoostRegressor(random_state=42, verbose=0, task_type="GPU", devices="0"),
    cat_params, n_iter=8, cv=3, scoring="r2", random_state=42, n_jobs=1
)
cat_search.fit(tune_sample, y_tune_sample)
print("Best CatBoost params:", cat_search.best_params_)

start = time.time()
cat_model = CatBoostRegressor(**cat_search.best_params_, random_state=42, verbose=0, task_type="GPU", devices="0")
cat_model.fit(X_train, y_train)
train_time = time.time() - start

cat_model = evaluate_model("CatBoost", cat_model, X_train, y_train, X_test, y_test, train_time)

### Compile results into a comparison table   

In [ ]:
results_df = pd.DataFrame(results).T.sort_values("R2", ascending=False)
results_df

#### Save all trained models    

In [ ]:
import os
os.makedirs("../models/trained", exist_ok=True)

models_dict = {
    "Linear Regression": lr,
    "Decision Tree": dt,
    "Random Forest": rf,
    "Gradient Boosting": gb,
    "XGBoost": xgb_model,
    "LightGBM": lgbm_model,
    "CatBoost": cat_model
}

for name, model in models_dict.items():
    filename = name.lower().replace(" ", "_")
    joblib.dump(model, f"../models/trained/{filename}.pkl")

print("All models saved to ../models/trained/")

###  Save results table

In [ ]:
results_df.to_csv("../reports/model_comparison.csv")

with open("../reports/model_training_summary.json", "w") as f:
    json.dump({
        "models_trained": list(models_dict.keys()),
        "best_model_by_r2": results_df.index[0],
        "best_r2_score": float(results_df.iloc[0]["R2"])
    }, f, indent=4)

print("Best model by R²:", results_df.index[0])
results_df